In [1]:
import numpy as np
import pandas as pd

# Task 0
Read the dataset from csv file & perform data cleaning - remove all rows, which contains `?` in some columns.
Also check for data correctness (salary & salary $K).

In [24]:
# write your code for task 0 here
df = pd.read_csv("../data/adult.csv", index_col=0)
print(df.shape)

has_question = df.astype(str).apply(lambda col: col.str.strip() == "?").any(axis=1)
print("Rows with '?':", has_question.sum())

df = df[~has_question]
print(df.shape)

df["salary"] = df["salary"].str.strip()
df["salary K$"] = pd.to_numeric(df["salary K$"])

inconsistent = (
    ((df["salary"] == ">50K") & (df["salary K$"] <= 50))
    | ((df["salary"] == "<=50K") & (df["salary K$"] > 50))
)
print("Inconsistent rows:", inconsistent.sum())
print(df[inconsistent])

df = df[~inconsistent]
print("Final shape:", df.shape)


(32561, 12)
Rows with '?': 2399
(30162, 12)
Inconsistent rows: 0
Empty DataFrame
Columns: [age, workclass, education, marital-status, occupation, relationship, race, sex, hours-per-week, native-country, salary, salary K$]
Index: []
Final shape: (30162, 12)


# Task 1
Print the count of men and women in the dataset.

In [26]:
# write your code for task 1 here
sex_counts = df["sex"].str.strip().value_counts()
sex_counts

sex
Male      20380
Female     9782
Name: count, dtype: int64

# Task 2
Find the average age of men in dataset

In [30]:
# write your code for task 2 here
avg_age_men = df[df["sex"].str.strip() == "Male"]["age"].mean()
avg_age_men

np.float64(39.18400392541707)

# Task 3
Get the percentage of people from Poland (native-country)

In [32]:
# write your code for task 3 here
polish_count = df[df["native-country"].str.strip() == "Poland"].shape[0]
total_count = df.shape[0]
percentage_polish = (polish_count / total_count) * 100
percentage_polish

0.18566408063125786

# Task 4
Get the mean and standard deviation of the age for people who earn > 50K per year. After this, get it for those who earn <= 50K.

In [34]:
# write your code for task 4 here
df.groupby("salary")["age"].agg(["mean", "std"])

,mean,std
salary,,
<=50K,36.60806,13.464631
>50K,43.95911,10.269633


# Task 5
Check, if there are some people without higher education (education: Bachelors, Prof-school, Assoc-acdm, Assoc-voc, Masters, Doctorate), but with > 50K salary

In [38]:
# write your code for task 5 here
higher_education = [
    "Bachelors",
    "Prof-school",
    "Assoc-acdm",
    "Assoc-voc",
    "Masters",
    "Doctorate",
]

education = df["education"].str.strip()
rich_without_higher = ~education.isin(higher_education) & (df["salary"] == ">50K")

rich_without_higher.any(), rich_without_higher.sum()

(np.True_, np.int64(3178))

# Task 6
Get the statistics of age for each type of education. Use `groupby` and `describe` for this.

In [40]:
# write your code for task 6 here
df["education"] = df["education"].str.strip()
df.groupby("education")["age"].describe().sort_values("mean")

,count,mean,std,min,25%,50%,75%,max
education,,,,,,,,
12th,377.0,32.013263,14.373710,17.0,19.0,28.0,41.0,79.0
11th,1048.0,32.363550,15.089307,17.0,18.0,28.5,43.0,90.0
Some-college,6678.0,36.135370,13.073528,17.0,25.0,35.0,45.0,90.0
Assoc-acdm,1008.0,37.286706,10.509755,19.0,29.0,36.0,44.0,90.0
10th,820.0,37.897561,16.225795,17.0,23.0,36.0,52.0,90.0
Assoc-voc,1307.0,38.246366,11.181253,19.0,30.0,37.0,45.0,84.0
HS-grad,9840.0,38.640955,13.067730,17.0,28.0,37.0,48.0,90.0
Bachelors,5044.0,38.641554,11.577566,19.0,29.0,37.0,46.0,90.0
9th,455.0,40.303297,15.335754,17.0,28.0,38.0,53.0,90.0


# Task 7
Compare the married and non-married men salaries. Who earns more? (>50K or <=50K)
Married men are those, whom `marital-status` starts with "Married". Others are not.

In [41]:
# write your code for task 7 here
men = df[df["sex"].str.strip() == "Male"]
is_married = men["marital-status"].str.strip().str.startswith("Married")

pd.crosstab(is_married, men["salary"], normalize="index", rownames=["married"]) * 100

salary,<=50K,>50K
married,,
False,91.150559,8.849441
True,55.201566,44.798434


# Task 8
Get the max hours per week some person works. How many people works the same amount of hours per week?

In [42]:
# write your code for task 8 here
max_hours = df["hours-per-week"].max()
people_with_max_hours = (df["hours-per-week"] == max_hours).sum()

max_hours, people_with_max_hours

(np.int64(99), np.int64(78))

# Task 9
Analyze the correlation between data in dataset. Understand connected fields in it and print highlight thier connection.

In [46]:
# write your code for task 9 here
corr = df.corr(numeric_only=True)
corr

,age,hours-per-week,salary K$
age,1.000000,0.101599,0.208203
hours-per-week,0.101599,1.000000,0.196378
salary K$,0.208203,0.196378,1.000000
